# 02 · Seleção de variáveis: pré-filtros e Boruta

**Etapas 2 e 3 do pipeline.** Recebe a base da etapa 1 (com a coluna `particao`) e a lista de variáveis candidatas, e devolve as variáveis que seguem para o AutoML.

| passo | onde roda | o que faz |
|---|---|---|
| 1 | Spark | classifica as variáveis em numéricas e categóricas |
| 2 | Spark | agrupa categorias raras em `OUTROS` (regra aprendida no treino) |
| 3 | Spark | corta cada variável em faixas (quantis do treino) e agrega tudo numa passada |
| 4 | Spark → tabela pequena | mede faltantes, valor dominante, PSI, poder univariado e suspeita de vazamento |
| 5 | pandas (amostra de 20 mil linhas do treino) | correlação de Spearman entre as variáveis numéricas (e com a LGD) |
| 6 | — | aplica os pré-filtros em ordem e registra o motivo de cada exclusão |
| 7 | pandas (treino ou amostra) | Boruta com LightGBM e teste binomial a 99% |
| 8 | — | funil de seleção e gráfico do Boruta |

**Tudo que é aprendido usa só o treino** (categorias mantidas, cortes das faixas, medianas, Boruta). O teste e o OOT só são *medidos* com essas regras. A exceção documentada é o PSI, que compara treino e OOT: é a medida de estabilidade da variável.

**Ordem dos pré-filtros.** Cada variável recebe o **primeiro** motivo que a exclui, nesta ordem:

| ordem | regra | PD | LGD |
|---|---|---|---|
| 1 | tipo suportado | número, texto ou booleano | igual |
| 2 | faltantes | > 95% vazia | igual |
| 3 | quase constante | um valor (faixa) com > 99% dos contratos | igual |
| 4 | estabilidade | PSI treino → OOT > 0,25 | igual |
| 5 | poder univariado | IV < 0,02 | \|Spearman\| < 0,05 (numéricas) ou η < 0,05 (categóricas) |
| 6 | teto da correlação | só as 1.500 de maior IV seguem | só as 1.500 de maior η seguem (vem antes do passo 5 na LGD, porque o Spearman com a LGD sai da correlação) |
| 7 | redundância | correlação de Spearman > 0,85 com uma variável mais forte | igual |
| 8 | teto do Boruta | só as 400 de maior IV seguem | só as 400 de maior força seguem |

A **suspeita de vazamento** não exclui: só marca a variável para análise humana.

**Funil para bases largas.** Os filtros 1 a 5 são baratos (custo linear) e rodam na base inteira. Os dois passos caros recebem menos: a correlação, cujo custo cresce com o quadrado do número de variáveis, recebe no máximo 1.500 variáveis e roda numa amostra de 20 mil linhas; o Boruta, que roda numa única máquina, recebe no máximo 400. Se um teto cortar uma variável de poder médio (IV ou η ≥ 0,1), o pipeline avisa: é sinal de que o teto está baixo para a base. Os três números são ajustáveis em `cfg["prefiltros"]`.

In [ ]:
%run ./00_config

In [ ]:
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

# Rótulo da categoria (e da faixa) de valores vazios.
ROTULO_FALTANTE = "FALTANTE"

# Constante numérica pequena que evita log(0) no PSI e no IV quando uma faixa
# não tem contratos numa das amostras (mesma convenção da biblioteca Yggdrasil).
EPSILON = 1e-6

# Tipos Spark aceitos como variável explicativa.
TIPOS_NUMERICOS = ("tinyint", "smallint", "int", "bigint", "float", "double")
TIPOS_CATEGORICOS = ("string",)

## 1. Tipos das variáveis

| tipo Spark | tratamento |
|---|---|
| inteiros, `float`, `double`, `decimal` | numérica |
| `boolean` | convertida para 0/1 e tratada como numérica |
| `string` | categórica |
| datas, listas, mapas | não suportada: excluída com motivo registrado |

In [ ]:
def classifica_tipos_variaveis(base, variaveis):
    """Devolve {variavel: "numerica" | "categorica" | "booleana" | "nao_suportada"}."""
    tipos_spark = dict(base.dtypes)
    tipos = {}
    for variavel in variaveis:
        tipo = tipos_spark[variavel]
        if tipo in TIPOS_NUMERICOS or tipo.startswith("decimal"):
            tipos[variavel] = "numerica"
        elif tipo == "boolean":
            tipos[variavel] = "booleana"
        elif tipo in TIPOS_CATEGORICOS:
            tipos[variavel] = "categorica"
        else:
            tipos[variavel] = "nao_suportada"
    return tipos


def padroniza_tipos(base, tipos):
    """
    Converte booleanas em 0/1 (inteiro) e decimais em double.

    Conceito técnico:
        As bibliotecas de modelo (pyspark.ml, LightGBM, XGBoost) esperam números
        de ponto flutuante; o tipo decimal do Spark é exato, mas lento e não é
        aceito por todos os algoritmos. Depois daqui, booleanas viram numéricas.
    """
    # Todas as conversões num único select. Um withColumn por coluna, em laço,
    # empilha uma camada no plano Spark por coluna; com milhares de colunas, o
    # plano fica grande demais (no serverless, a conexão desiste: RETRIES_EXCEEDED).
    tipos_spark = dict(base.dtypes)
    convertidas = {}
    for variavel, tipo in list(tipos.items()):
        if tipo == "booleana":
            convertidas[variavel] = F.col(variavel).cast("int")
            tipos[variavel] = "numerica"
        elif tipo == "numerica" and tipos_spark[variavel].startswith("decimal"):
            convertidas[variavel] = F.col(variavel).cast("double")
    if convertidas:
        base = base.select(*[convertidas[c].alias(c) if c in convertidas else F.col(c)
                             for c in base.columns])
    return base, tipos

## 2. Agrupamento de categorias raras

Para cada variável categórica, mede a participação de cada categoria **no treino**:

- categoria com ≥ 1% dos contratos do treino é mantida;
- categoria com < 1% vai para `OUTROS`;
- valor vazio vira a categoria `FALTANTE`, que segue a mesma regra;
- categoria que aparece no teste, no OOT ou no HIST e não existia no treino também vai para `OUTROS`.

**Exemplo.** Variável `uf`, treino com 100.000 contratos: SP 41.000, RJ 18.000, MG 15.000, … , AC 300 (0,3%). AC vira `OUTROS`. Uma UF que só aparece no OOT também vira `OUTROS`, e o modelo trata os dois casos do mesmo jeito.

A regra (categorias mantidas por variável) é devolvida em `mapa_categorias`, para ser reaplicada idêntica na escoragem de bases novas.

In [ ]:
def agrupa_categorias_raras(base, variaveis_categoricas, cfg):
    """
    Aprende no treino quais categorias manter e aplica a regra em toda a base.

    Devolve
        (base_agrupada, mapa_categorias, tabela_categorias)
        mapa_categorias   {variavel: [categorias mantidas]}
        tabela_categorias pandas: variavel, categoria, contratos_treino,
                          participacao_treino, destino ("mantida" | "OUTROS")

    Conceito de negócio:
        Uma categoria com poucos contratos tem taxa de inadimplência instável:
        30 contratos com 3 maus dão 10%, com 1 mau dão 3,3%. O modelo aprenderia
        ruído. Juntar as raras numa categoria só dá volume para estimar o risco.
    """
    if not variaveis_categoricas:
        return base, {}, pd.DataFrame(columns=["variavel", "categoria", "contratos_treino",
                                               "participacao_treino", "destino"])
    piso = cfg["preprocessamento"]["piso_categoria_rara"]
    rotulo_outros = cfg["preprocessamento"]["rotulo_categoria_rara"]

    # Formato longo (variavel, categoria) numa passada pelo treino.
    treino = base.filter(F.col("particao") == "treino")
    colunas_texto = [F.coalesce(F.col(v).cast("string"), F.lit(ROTULO_FALTANTE)).alias(v)
                     for v in variaveis_categoricas]
    contagem = (treino.select(*colunas_texto)
                      .unpivot([], variaveis_categoricas, "variavel", "categoria")
                      .groupBy("variavel", "categoria").count()
                      .toPandas())
    contagem["participacao_treino"] = (contagem["count"]
                                       / contagem.groupby("variavel")["count"].transform("sum"))
    contagem["destino"] = np.where(contagem["participacao_treino"] >= piso, "mantida", rotulo_outros)
    contagem = (contagem.rename(columns={"count": "contratos_treino"})
                        .sort_values(["variavel", "contratos_treino"], ascending=[True, False])
                        .reset_index(drop=True))

    mapa_categorias = {
        v: sorted(contagem.loc[(contagem["variavel"] == v) & (contagem["destino"] == "mantida"),
                               "categoria"].tolist())
        for v in variaveis_categoricas
    }
    novas = {}
    for variavel, mantidas in mapa_categorias.items():
        valor = F.coalesce(F.col(variavel).cast("string"), F.lit(ROTULO_FALTANTE))
        novas[variavel] = (F.when(valor.isin(*mantidas), valor).otherwise(F.lit(rotulo_outros))
                           if mantidas else F.lit(rotulo_outros))
    base = base.withColumns(novas)          # todas as colunas numa única camada do plano
    return base, mapa_categorias, contagem

## 3. Faixas e agregação numa passada

Cada variável é cortada em faixas para medir PSI, IV e o valor dominante:

- **numérica:** cortes nos decis do **treino** (quantis 10%, 20%, …, 90%, calculados com `approxQuantile`, que estima quantis no Spark sem ordenar a base inteira). Cortes repetidos são unidos. Vazio (e `NaN`) vira a faixa `FALTANTE`;
- **categórica:** a própria categoria, já agrupada.

Depois, a base é posta em **formato longo** (uma linha por contrato e variável, com a faixa como texto) e agregada uma única vez por variável, faixa e partição: contratos, soma do alvo e soma do alvo ao quadrado. O resultado tem poucas centenas de linhas por variável e vai para pandas.

A mesma passada pelo `approxQuantile` devolve a **mediana do treino** de cada variável numérica, usada para preencher vazios na correlação (passo 5) e, depois, nas técnicas que não aceitam vazios.

In [ ]:
def calcula_cortes_e_medianas(base, variaveis_numericas, n_faixas, erro_relativo=0.001, tamanho_lote=500):
    """
    Cortes das faixas (decis do treino) e mediana do treino, por variável numérica.

    Devolve
        (cortes, medianas)  cortes: {variavel: [c1 < c2 < ...]}; medianas: {variavel: valor}

    Conceito técnico:
        approxQuantile calcula todos os quantis de todas as colunas numa só
        passada, com erro relativo de 0,1% na posição do quantil. Ignora vazios.
        Variável toda vazia no treino devolve lista vazia: fica sem cortes e
        com mediana indefinida (é excluída depois pelo filtro de faltantes).
    """
    if not variaveis_numericas:
        return {}, {}
    probabilidades = [k / n_faixas for k in range(1, n_faixas)] + [0.5]
    quantis = []
    for lote in em_lotes(variaveis_numericas, tamanho_lote):
        treino = base.filter(F.col("particao") == "treino").select(
            *[F.when(F.isnan(F.col(v).cast("double")), None).otherwise(F.col(v).cast("double")).alias(v)
              for v in lote])
        quantis += treino.approxQuantile(lote, probabilidades, erro_relativo)
    cortes, medianas = {}, {}
    for variavel, valores in zip(variaveis_numericas, quantis):
        if not valores:
            cortes[variavel], medianas[variavel] = [], None
            continue
        cortes[variavel] = sorted(set(valores[:-1]))
        medianas[variavel] = valores[-1]
    return cortes, medianas


def expressao_faixa(variavel, tipo, cortes):
    """
    Coluna Spark com a faixa de cada contrato, como texto ("00", "01", ..., "FALTANTE").

    Faixa "00" = até o 1º corte; faixa "k" = entre o corte k e o k+1; a última
    faixa vai do último corte em diante. Para categóricas, a faixa é a categoria.
    """
    if tipo == "categorica":
        return F.col(variavel).cast("string")
    valor = F.col(variavel).cast("double")
    expressao = F.when(valor.isNull() | F.isnan(valor), F.lit(ROTULO_FALTANTE))
    for indice, corte in enumerate(cortes):
        expressao = expressao.when(valor <= F.lit(float(corte)), F.lit(f"{indice:02d}"))
    return expressao.otherwise(F.lit(f"{len(cortes):02d}"))


def agrega_por_faixa(base, tipos, cortes, cfg):
    """
    Uma linha por (variavel, faixa, particao): contratos, soma do alvo e soma do
    alvo ao quadrado. Só treino, teste e OOT (o HIST não entra na seleção).

    Conceito técnico:
        unpivot transforma as N colunas de faixa em linhas (variavel, faixa). Com
        isso, um único groupBy agrega todas as variáveis, em vez de N jobs.
    """
    alvo = cfg["colunas"]["alvo"]
    variaveis = [v for v, t in tipos.items() if t in ("numerica", "categorica")]
    partes = []
    # Em lotes de variáveis: o formato longo de um lote tem (linhas × tamanho do
    # lote) registros, em vez de (linhas × todas as variáveis) de uma vez.
    for lote in em_lotes(variaveis, cfg["tamanho_lote_variaveis"]):
        selecao = (base.filter(F.col("particao").isin("treino", "teste", "oot"))
                       .select("particao", F.col(alvo).cast("double").alias("_alvo"),
                               *[expressao_faixa(v, tipos[v], cortes.get(v, [])).alias(v)
                                 for v in lote]))
        partes.append(selecao.unpivot(["particao", "_alvo"], lote, "variavel", "faixa")
                             .groupBy("variavel", "faixa", "particao")
                             .agg(F.count(F.lit(1)).alias("contratos"),
                                  F.sum("_alvo").alias("soma_alvo"),
                                  F.sum(F.col("_alvo") * F.col("_alvo")).alias("soma_alvo_quadrado"))
                             .toPandas())
    return pd.concat(partes, ignore_index=True) if partes else pd.DataFrame(
        columns=["variavel", "faixa", "particao", "contratos", "soma_alvo", "soma_alvo_quadrado"])

## 4. Métricas univariadas

Calculadas sobre a tabela agregada (pandas, pequena):

| métrica | conta | leitura |
|---|---|---|
| `pct_faltante` | contratos na faixa `FALTANTE` / total, no treino | quanto da variável é vazio |
| `participacao_dominante` | maior participação de uma faixa no treino | perto de 100% = quase constante |
| `psi` | Σ (a − e) · ln(a / e), com e = participação no treino e a = no OOT | quanto a distribuição mudou: < 0,10 estável; 0,10 a 0,25 atenção; > 0,25 instável |
| `iv` (PD) | Σ (% dos maus − % dos bons) · ln(% dos maus / % dos bons) | poder de separar bons e maus: < 0,02 inútil; 0,02 a 0,1 fraco; 0,1 a 0,3 médio; > 0,3 forte |
| `ks_faixas` (PD) | KS calculado sobre as faixas | KS univariado aproximado (com faixas, fica um pouco abaixo do KS contínuo) |
| `eta` (LGD) | √(variância da LGD entre faixas / variância total) | razão de correlação (η): quanto as faixas explicam a LGD; serve para numéricas e categóricas |

**Exemplo de IV.** Comprometimento em 2 faixas no treino (9.400 bons e 600 maus):

```
até 30%:   bons 61,3%   maus 40,0%   → (0,400 − 0,613) · ln(0,400 / 0,613) = 0,091
acima:     bons 38,7%   maus 60,0%   → (0,600 − 0,387) · ln(0,600 / 0,387) = 0,093
IV = 0,18  → poder médio
```

In [ ]:
def _psi(participacao_esperada, participacao_atual):
    """PSI entre duas distribuições (Series indexadas pela faixa)."""
    faixas = participacao_esperada.index.union(participacao_atual.index)
    e = participacao_esperada.reindex(faixas, fill_value=0.0)
    a = participacao_atual.reindex(faixas, fill_value=0.0)
    return float(((a - e) * np.log((a + EPSILON) / (e + EPSILON))).sum())


def _iv_e_ks(faixas_treino):
    """
    IV e KS a partir de contratos e maus por faixa (PD).

    Conceito técnico:
        Para o KS, as faixas são ordenadas pela taxa de maus; o KS é a maior
        distância entre a % acumulada de maus e a % acumulada de bons.
    """
    maus = faixas_treino["soma_alvo"]
    bons = faixas_treino["contratos"] - maus
    if maus.sum() == 0 or bons.sum() == 0:
        return 0.0, 0.0
    pct_maus, pct_bons = maus / maus.sum(), bons / bons.sum()
    iv = float(((pct_maus - pct_bons) * np.log((pct_maus + EPSILON) / (pct_bons + EPSILON))).sum())
    ordem = (maus / faixas_treino["contratos"]).sort_values().index
    ks = float((pct_maus[ordem].cumsum() - pct_bons[ordem].cumsum()).abs().max())
    return iv, ks


def _eta(faixas_treino):
    """
    Razão de correlação η (LGD): parte da variação da LGD explicada pelas faixas.

    η² = Σ n_f · (média_f − média)² / Σ (y − média)²  — calculado só com contagem,
    soma e soma dos quadrados de cada faixa (não precisa das linhas).
    """
    n = faixas_treino["contratos"]
    soma = faixas_treino["soma_alvo"]
    total, media = n.sum(), soma.sum() / n.sum()
    variacao_total = faixas_treino["soma_alvo_quadrado"].sum() - total * media ** 2
    if variacao_total <= 0:
        return 0.0
    variacao_entre = (n * (soma / n - media) ** 2).sum()
    return float(np.sqrt(max(variacao_entre, 0.0) / variacao_total))


def calcula_metricas_univariadas(agregado, tipos, tarefa):
    """Uma linha por variável com as métricas da tabela acima."""
    linhas = []
    for variavel, grupo in agregado.groupby("variavel"):
        treino = grupo[grupo["particao"] == "treino"].set_index("faixa")
        oot = grupo[grupo["particao"] == "oot"].set_index("faixa")
        participacao_treino = treino["contratos"] / treino["contratos"].sum()
        participacao_oot = (oot["contratos"] / oot["contratos"].sum()) if len(oot) else participacao_treino
        linha = {
            "variavel": variavel,
            "tipo": tipos[variavel],
            "n_faixas": int(len(treino)),
            "pct_faltante": float(participacao_treino.get(ROTULO_FALTANTE, 0.0)),
            "participacao_dominante": float(participacao_treino.max()),
            "psi": _psi(participacao_treino, participacao_oot),
        }
        if tarefa == "pd":
            linha["iv"], linha["ks_faixas"] = _iv_e_ks(treino)
        else:
            linha["eta"] = _eta(treino)
        linhas.append(linha)
    return pd.DataFrame(linhas)

## 5. Correlação de Spearman

A correlação de Spearman é a correlação entre as **ordens** dos valores: mede se uma variável sobe quando a outra sobe, sem exigir relação em linha reta, e não é distorcida por valores extremos. É calculada no pandas, numa **amostra aleatória do treino** (20 mil linhas por padrão), só para as variáveis numéricas que passaram pelos filtros anteriores e pelo teto, com os vazios preenchidos pela mediana do treino. Conta: cada coluna vira o seu ranking, e a correlação comum entre rankings é a de Spearman.

**Por que a amostra basta.** O erro de uma correlação estimada em n linhas é aproximadamente (1 − ρ²) / √n. No limite de redundância, ρ = 0,85: (1 − 0,7225) / √20.000 = 0,2775 / 141 ≈ 0,002. Uma correlação de 0,85 na base inteira sai entre cerca de 0,846 e 0,854 na amostra; a decisão só muda para pares praticamente sobre o limite. O custo cresce com linhas × variáveis², então a amostra e o teto juntos reduzem a conta em mais de 100 vezes numa base com milhares de variáveis.

- Entre variáveis: alimenta o filtro de redundância.
- Com a LGD (só na tarefa LGD): é o poder univariado das variáveis numéricas.

**Exemplo de redundância.** `saldo_medio_3m` e `saldo_medio_6m` com Spearman de 0,93. As duas carregam quase a mesma informação. Fica a de maior poder (IV ou \|Spearman\| com a LGD), e a outra sai com o motivo "redundante com saldo_medio_3m (0,93)".

As variáveis categóricas não entram no filtro de redundância.

In [ ]:
def correlacao_spearman(base, variaveis_numericas, medianas, cfg, coluna_alvo=None):
    """
    Matriz de Spearman (pandas) entre as variáveis numéricas, numa amostra
    aleatória do treino de cfg["prefiltros"]["n_linhas_correlacao"] linhas.
    Com `coluna_alvo`, o alvo entra como última coluna da matriz.

    Conceito técnico:
        Spearman = correlação de Pearson entre os rankings. Cada coluna é
        trocada pela sua posição (empates recebem a posição média) e a matriz
        sai de np.corrcoef. Vazios são preenchidos pela mediana do treino.
    """
    colunas = list(variaveis_numericas) + ([coluna_alvo] if coluna_alvo else [])
    if len(colunas) < 2:
        return pd.DataFrame(np.eye(len(colunas)), index=colunas, columns=colunas)
    preenchidas = []
    for coluna in colunas:
        valor = F.col(coluna).cast("double")
        mediana = medianas.get(coluna, 0.0) if coluna != coluna_alvo else 0.0
        preenchidas.append(F.when(valor.isNull() | F.isnan(valor), F.lit(float(mediana or 0.0)))
                            .otherwise(valor).alias(coluna))
    treino = base.filter(F.col("particao") == "treino")
    n_treino = treino.count()
    n_alvo = cfg["prefiltros"]["n_linhas_correlacao"]
    fracao = min(1.0, n_alvo / max(n_treino, 1))
    amostra = (treino.sample(withReplacement=False, fraction=fracao, seed=cfg["semente"])
                     .select(*preenchidas).toPandas())
    print(f"[seleção] correlação de Spearman: {len(colunas)} variáveis em {len(amostra):,} linhas do treino")
    rankings = amostra.astype("float64").rank(method="average").to_numpy()
    with np.errstate(invalid="ignore", divide="ignore"):
        matriz = np.corrcoef(rankings, rowvar=False)
    # Variável constante na amostra gera correlação indefinida (NaN): tratada como 0.
    matriz = np.nan_to_num(np.atleast_2d(matriz), nan=0.0)
    return pd.DataFrame(matriz, index=colunas, columns=colunas)

## 6. Aplicação dos pré-filtros

As regras são aplicadas na ordem da tabela do início. A **suspeita de vazamento** é marcada em paralelo:

- PD: IV > 0,5 ou KS das faixas > 0,5;
- LGD: \|Spearman\| > 0,7 ou η > 0,7.

Vazamento é uma variável que carrega informação do futuro, por exemplo um campo atualizado depois da concessão. O sintoma é poder preditivo bom demais. Uma variável legitimamente muito forte (um score de *bureau*, por exemplo) também pode disparar o aviso, por isso a decisão fica com quem analisa.

In [ ]:
def aplica_prefiltros(base, tipos, cortes, medianas, cfg):
    """
    Executa os passos 3 a 6 e devolve a tabela de pré-filtros (pandas), com uma
    linha por variável: métricas, suspeita_vazamento, status e motivo.
    """
    p = cfg["prefiltros"]
    tarefa = cfg["tarefa"]

    agregado = agrega_por_faixa(base, tipos, cortes, cfg)
    tabela = calcula_metricas_univariadas(agregado, tipos, tarefa)
    nao_suportadas = [v for v, t in tipos.items() if t == "nao_suportada"]
    if nao_suportadas:
        tabela = pd.concat([tabela, pd.DataFrame({"variavel": nao_suportadas,
                                                  "tipo": "nao_suportada"})], ignore_index=True)
    tabela["status"], tabela["motivo"] = "mantida", ""

    def descarta(mascara, motivo):
        alvo = mascara & (tabela["status"] == "mantida")
        tabela.loc[alvo, "status"] = "descartada"
        tabela.loc[alvo, "motivo"] = motivo

    descarta(tabela["tipo"] == "nao_suportada", "tipo não suportado")
    descarta(tabela["pct_faltante"] > p["max_faltantes"], f"faltantes > {p['max_faltantes']:.0%}")
    descarta(tabela["participacao_dominante"] > p["max_participacao_valor_dominante"],
             f"quase constante (> {p['max_participacao_valor_dominante']:.0%} numa faixa)")
    descarta(tabela["psi"] > p["max_psi"], f"instável (PSI treino→OOT > {p['max_psi']})")

    def aplica_teto(maximo, forca, rotulo):
        """Mantém só as `maximo` variáveis vivas de maior força; avisa se cortar alguma forte."""
        vivas = tabela.index[tabela["status"] == "mantida"]
        if len(vivas) <= maximo:
            return
        ordem = forca.loc[vivas].fillna(0).sort_values(ascending=False)
        cortadas = ordem.index[maximo:]
        descarta(tabela.index.isin(cortadas), f"{rotulo}: fora das {maximo} de maior força")
        forca_max_cortada = float(ordem.iloc[maximo])
        print(f"[seleção] {rotulo}: {len(cortadas)} variáveis cortadas (maior força cortada = "
              f"{forca_max_cortada:.3f}); seguem {maximo}.")
        if forca_max_cortada >= p["forca_alerta_teto"]:
            print(f"[seleção] ATENÇÃO — o {rotulo} cortou variáveis com força ≥ {p['forca_alerta_teto']} "
                  f"(poder médio). Considere aumentar o teto em cfg['prefiltros'].")

    # Poder univariado (PD) e teto 1, antes da correlação. Na LGD o teto vem
    # primeiro, pela η, porque o Spearman com a LGD sai da própria correlação.
    if tarefa == "pd":
        tabela["forca"] = tabela["iv"]
        descarta(tabela["iv"] < p["min_iv"], f"sem poder (IV < {p['min_iv']})")
        tabela["suspeita_vazamento"] = (tabela["iv"] > p["vazamento_iv"]) | (tabela["ks_faixas"] > p["vazamento_ks"])
        aplica_teto(p["max_variaveis_correlacao"], tabela["iv"], "teto da correlação")
        numericas_vivas = tabela.loc[(tabela["status"] == "mantida") & (tabela["tipo"] == "numerica"),
                                     "variavel"].tolist()
        matriz = correlacao_spearman(base, numericas_vivas, medianas, cfg)
    else:
        alvo = cfg["colunas"]["alvo"]
        aplica_teto(p["max_variaveis_correlacao"], tabela["eta"], "teto da correlação")
        numericas_vivas = tabela.loc[(tabela["status"] == "mantida") & (tabela["tipo"] == "numerica"),
                                     "variavel"].tolist()
        matriz_completa = correlacao_spearman(base, numericas_vivas, medianas, cfg, coluna_alvo=alvo)
        spearman_alvo = matriz_completa[alvo].drop(alvo)
        tabela["spearman_alvo"] = tabela["variavel"].map(spearman_alvo)
        # Força: |Spearman| para numéricas; η para categóricas (Spearman não se aplica a texto).
        tabela["forca"] = np.where(tabela["tipo"] == "numerica",
                                   tabela["spearman_alvo"].abs(), tabela["eta"])
        descarta(tabela["forca"].fillna(0) < p["min_spearman_alvo"],
                 f"sem poder (|Spearman| ou η < {p['min_spearman_alvo']})")
        tabela["suspeita_vazamento"] = ((tabela["spearman_alvo"].abs() > p["vazamento_spearman"])
                                        | (tabela["eta"] > p["vazamento_spearman"]))
        matriz = matriz_completa.drop(index=alvo, columns=alvo)
    tabela["suspeita_vazamento"] = tabela["suspeita_vazamento"].fillna(False).astype(bool)

    # Redundância: percorre as numéricas da mais forte para a mais fraca; cada uma
    # fica se não tiver correlação > limite com alguma que já ficou.
    candidatas = (tabela[(tabela["status"] == "mantida") & (tabela["tipo"] == "numerica")]
                  .sort_values("forca", ascending=False)["variavel"].tolist())
    mantidas = []
    for variavel in candidatas:
        conflito = [(m, abs(matriz.loc[variavel, m])) for m in mantidas
                    if abs(matriz.loc[variavel, m]) > p["max_correlacao"]]
        if conflito:
            mais_forte, correlacao = max(conflito, key=lambda par: par[1])
            descarta(tabela["variavel"] == variavel,
                     f"redundante com {mais_forte} (Spearman {correlacao:.2f})")
        else:
            mantidas.append(variavel)

    # Teto 2: só as M mais fortes seguem para o Boruta (roda numa única máquina).
    aplica_teto(p["max_variaveis_boruta"], tabela["forca"], "teto do Boruta")

    tabela["etapa"] = np.where(tabela["status"] == "mantida", "pre_filtros_aprovada", "pre_filtros")
    return tabela.sort_values(["status", "forca"], ascending=[False, False]).reset_index(drop=True)

## 7. Boruta com LightGBM

**Amostra.** Sai **só do treino**, estratificada por safra (sorteada separadamente em cada grupo):

| tarefa | amostra | estratos |
|---|---|---|
| PD (padrão: caso-controle) | **todos os maus** (até 50.000) e **9 bons sorteados para cada mau** | safra × alvo |
| PD (opção `"formula"`) | n = z² · p · (1 − p) / e², com z = 2,576 (99%), p = taxa de maus, e = 2% de p | safra × alvo |
| LGD | treino inteiro se tiver até 300.000 contratos; senão n = z² · σ² / e², com σ = desvio-padrão da LGD e e = 2% da média | safra × faixa da LGD (0, entre 0 e 1, 1) |

**Por que caso-controle na PD.** O Boruta não estima a taxa de maus: compara a importância de cada variável com a das sombras. A informação dessa comparação vem dos maus. Com evento raro, a fórmula de confiança pede mais linhas do que o treino tem e acaba usando o treino inteiro, quase todo de bons, que pouco acrescentam.

**Exemplo.** Treino com 94.807 contratos e taxa de 1,49%:

```
fórmula:        e = 0,02 × 0,0149 = 0,0003  →  n = 2,576² × 0,0149 × 0,9851 / 0,0003² ≈ 1,09 milhão  →  treino inteiro (94.807)
caso-controle:  maus = 0,0149 × 94.807 ≈ 1.413 (todos)
                bons = 9 × 1.413 = 12.717 (de 93.394: fração de 13,6%)
                total ≈ 14.130 linhas, com a mesma informação dos maus e 1/7 das linhas
```

A taxa de maus da amostra (10%) é maior que a do treino (1,49%), o que não afeta o Boruta: ele não usa a taxa, só a ordem de importância entre variáveis e sombras.

**As rodadas.** Em cada uma das 50 rodadas:

1. cria uma **sombra** de cada variável: uma cópia com os valores embaralhados entre os contratos, ou seja, puro ruído com a mesma distribuição;
2. treina um LightGBM **regularizado** com as variáveis reais e as sombras juntas;
3. mede a importância de cada variável pela **média do |SHAP|**: quanto, em média, ela desloca a previsão de um contrato. O SHAP é calculado pelo próprio LightGBM (`pred_contrib=True`) em 5.000 contratos sorteados;
4. a variável real ganha uma **vitória** se a sua importância superar a da melhor sombra.

**Por que SHAP e não ganho.** O ganho (quanto a variável reduziu o erro nas árvores) favorece variáveis contínuas, que oferecem muitos pontos de corte. Quando uma variável forte absorve a maior parte do sinal, o modelo passa a ajustar ruído, e as sombras contínuas acumulam ganho. No teste deste notebook, uma categórica informativa (IV de 0,12) perdeu para as sombras em **todas** as rodadas com ganho, e venceu em **todas** com SHAP e o LightGBM regularizado. A regularização (folha mínima de 1% da amostra, 15 folhas por árvore, 80% dos contratos e das variáveis por árvore) reduz o ajuste ao ruído, que é o que dá importância às sombras. As duas escolhas ficam na configuração (`boruta.medida_importancia`, `boruta.hiperparametros_lightgbm`).

**A decisão (teste binomial).** Se a variável fosse ruído, cada rodada seria como jogar uma moeda (50% de chance de vencer). O teste calcula a chance de o placar observado acontecer por sorte. Com 99% de confiança e a correção de Bonferroni (dividir 1% pelo número de variáveis testadas):

- **confirmada:** chance de tantas vitórias por sorte ≤ 1% / m → segue para o AutoML;
- **rejeitada:** chance de tão poucas vitórias por sorte ≤ 1% / m;
- **tentativa:** nenhum dos dois. Por decisão do desenho, **não segue**.

Com 150 variáveis e 50 rodadas: confirmada com ≥ 39 vitórias; rejeitada com ≤ 11.

In [ ]:
def tamanho_amostra_boruta(cfg, n_treino, media_alvo, desvio_alvo):
    """
    Tamanho da amostra do Boruta e se o treino inteiro deve ser usado.

    Devolve (n_amostra, usa_treino_inteiro, descricao_da_conta).
    """
    from scipy.stats import norm
    b = cfg["boruta"]
    z = float(norm.ppf(1 - (1 - b["confianca_amostra"]) / 2))   # 2,576 para 99%
    margem = b["margem_relativa"] * media_alvo
    if cfg["tarefa"] == "pd":
        n = z ** 2 * media_alvo * (1 - media_alvo) / margem ** 2
        conta = (f"z={z:.3f}, p={media_alvo:.4f}, e={margem:.5f} → "
                 f"n = z²·p·(1−p)/e² = {n:,.0f}")
        usa_inteiro = n >= n_treino
    else:
        n = z ** 2 * desvio_alvo ** 2 / margem ** 2
        conta = (f"z={z:.3f}, σ={desvio_alvo:.4f}, média={media_alvo:.4f}, e={margem:.5f} → "
                 f"n = z²·σ²/e² = {n:,.0f}")
        usa_inteiro = n_treino <= b["limite_treino_inteiro"] or n >= n_treino
    return int(np.ceil(min(n, n_treino))), bool(usa_inteiro), conta


def amostra_para_boruta(base, variaveis, tipos, mapa_categorias, cfg):
    """
    Sorteia a amostra estratificada do treino e devolve (X, y, info) em pandas.

    Conceito técnico:
        sampleBy sorteia cada contrato com a mesma probabilidade (fração) dentro
        de cada estrato. O tamanho final fica próximo do pedido (é um sorteio,
        não um corte exato), e a proporção de cada estrato é preservada.
    """
    colunas, semente = cfg["colunas"], cfg["semente"]
    alvo, safra = colunas["alvo"], colunas["safra"]
    treino = base.filter(F.col("particao") == "treino")
    resumo = treino.agg(F.count(F.lit(1)).alias("n"), F.avg(alvo).alias("media"),
                        F.stddev(alvo).alias("desvio")).collect()[0]
    n_amostra, usa_inteiro, conta = tamanho_amostra_boruta(
        cfg, resumo["n"], float(resumo["media"]), float(resumo["desvio"] or 0.0))

    b = cfg["boruta"]
    if cfg["tarefa"] == "pd" and b["amostragem_pd"] == "caso_controle":
        # Todos os maus (até max_maus) + razao_bons_por_mau bons para cada mau.
        n_maus = float(resumo["media"]) * resumo["n"]
        n_bons = resumo["n"] - n_maus
        fracao_maus = min(1.0, b["max_maus"] / max(n_maus, 1.0))
        fracao_bons = min(1.0, b["razao_bons_por_mau"] * n_maus * fracao_maus / max(n_bons, 1.0))
        treino = treino.withColumn("_estrato", F.concat_ws("|", F.col(safra).cast("string"),
                                                           F.col(alvo).cast("int").cast("string")))
        estratos = [linha["_estrato"] for linha in treino.select("_estrato").distinct().collect()]
        fracoes = {e: (fracao_maus if e.endswith("|1") else fracao_bons) for e in estratos}
        amostra = treino.sampleBy("_estrato", fracoes, seed=semente)
        usa_inteiro = fracao_maus >= 1.0 and fracao_bons >= 1.0
        n_amostra = int(round(n_maus * fracao_maus + n_bons * fracao_bons))
        conta = (f"caso-controle: {n_maus * fracao_maus:,.0f} maus ({fracao_maus:.0%}) + "
                 f"{n_bons * fracao_bons:,.0f} bons ({fracao_bons:.1%} dos {n_bons:,.0f}) "
                 f"≈ {n_amostra:,} linhas")
    elif usa_inteiro:
        amostra = treino
    else:
        if cfg["tarefa"] == "pd":
            grupo_alvo = F.col(alvo).cast("int").cast("string")
        else:
            grupo_alvo = (F.when(F.col(alvo) <= 0, "zero").when(F.col(alvo) >= 1, "um")
                           .otherwise("intermediaria"))
        treino = treino.withColumn("_estrato", F.concat_ws("|", F.col(safra).cast("string"), grupo_alvo))
        estratos = [linha["_estrato"] for linha in treino.select("_estrato").distinct().collect()]
        fracao = min(1.0, n_amostra / resumo["n"])
        amostra = treino.sampleBy("_estrato", {e: fracao for e in estratos}, seed=semente)

    pdf = amostra.select(*variaveis, F.col(alvo).cast("double").alias("_alvo")).toPandas()
    y = pdf.pop("_alvo")
    for variavel in variaveis:
        if tipos[variavel] == "categorica":
            # Categorias fixas = as mantidas no treino + OUTROS: as sombras e o
            # modelo enxergam o mesmo conjunto de categorias.
            categorias = mapa_categorias[variavel] + [cfg["preprocessamento"]["rotulo_categoria_rara"]]
            pdf[variavel] = pd.Categorical(pdf[variavel], categories=sorted(set(categorias)))
        else:
            pdf[variavel] = pd.to_numeric(pdf[variavel], errors="coerce").astype("float64")
    info = {"n_treino": int(resumo["n"]), "n_amostra_calculado": n_amostra,
            "usa_treino_inteiro": usa_inteiro, "n_amostra_realizado": int(len(pdf)),
            "conta_tamanho": conta}
    return pdf, y, info


def limiares_binomiais(n_rodadas, n_variaveis, nivel_significancia):
    """Menor nº de vitórias para confirmar e maior nº para rejeitar (com Bonferroni)."""
    from scipy.stats import binom
    nivel_corrigido = nivel_significancia / max(n_variaveis, 1)
    confirma = next((h for h in range(n_rodadas + 1)
                     if binom.sf(h - 1, n_rodadas, 0.5) <= nivel_corrigido), None)
    rejeita = next((h for h in range(n_rodadas, -1, -1)
                    if binom.cdf(h, n_rodadas, 0.5) <= nivel_corrigido), None)
    return confirma, rejeita, nivel_corrigido


def executa_boruta(X, y, cfg):
    """
    Roda as rodadas do Boruta e devolve a tabela de decisão (pandas).

    Colunas: variavel, vitorias, n_rodadas, taxa_vitorias, importancia_media,
             p_valor_confirmacao, p_valor_rejeicao, decisao.

    Conceito técnico:
        - Importância = média do |SHAP| (padrão) ou ganho, conforme
          cfg["boruta"]["medida_importancia"]. O SHAP vem do próprio LightGBM
          (pred_contrib=True: TreeSHAP exato, sem a biblioteca shap), calculado
          numa amostra de linhas por rodada para limitar o custo.
        - Folha mínima = participacao_minima_folha × nº de linhas, aplicada a
          min_child_samples (folhas) e min_data_per_group (grupos de categorias).
        - As sombras são embaralhadas de novo a cada rodada, com semente
          derivada da semente da configuração (reprodutível).
        - PD usa objetivo "binary"; LGD usa "cross_entropy" (alvo em [0, 1]).
    """
    import lightgbm as lgb
    from scipy.stats import binom

    b = cfg["boruta"]
    n_rodadas, variaveis = b["n_rodadas"], list(X.columns)
    eh_pd = cfg["tarefa"] == "pd"
    usa_shap = b["medida_importancia"] == "shap"
    folha_minima = max(20, int(round(b["participacao_minima_folha"] * len(X))))
    vitorias = pd.Series(0, index=variaveis)
    importancia_acumulada = pd.Series(0.0, index=variaveis)

    for rodada in range(n_rodadas):
        gerador = np.random.default_rng(cfg["semente"] + rodada)
        sombras = pd.DataFrame(
            {f"sombra_{v}": (pd.Categorical(gerador.permutation(X[v].to_numpy()),
                                            categories=X[v].cat.categories)
                             if isinstance(X[v].dtype, pd.CategoricalDtype)
                             else gerador.permutation(X[v].to_numpy()))
             for v in variaveis}, index=X.index)
        completo = pd.concat([X, sombras], axis=1)
        parametros = dict(b["hiperparametros_lightgbm"], random_state=cfg["semente"] + rodada,
                          min_child_samples=folha_minima, min_data_per_group=folha_minima,
                          importance_type="gain", n_jobs=-1, verbose=-1)
        if eh_pd:
            modelo = lgb.LGBMClassifier(objective="binary", **parametros)
            modelo.fit(completo, y.astype(int))
        else:
            modelo = lgb.LGBMRegressor(objective="cross_entropy", **parametros)
            modelo.fit(completo, y)

        if usa_shap:
            # Contribuições SHAP: uma coluna por variável + a última (valor base),
            # que é descartada. Importância = média do valor absoluto.
            linhas = completo.sample(n=min(b["n_linhas_shap"], len(completo)),
                                     random_state=cfg["semente"] + rodada)
            contribuicoes = modelo.booster_.predict(linhas, pred_contrib=True)[:, :-1]
            importancia = pd.Series(np.abs(contribuicoes).mean(axis=0), index=completo.columns)
        else:
            importancia = pd.Series(modelo.feature_importances_, index=completo.columns)

        melhor_sombra = importancia[[f"sombra_{v}" for v in variaveis]].max()
        vitorias += (importancia[variaveis] > melhor_sombra).astype(int)
        importancia_acumulada += importancia[variaveis]
        if (rodada + 1) % 10 == 0 or rodada + 1 == n_rodadas:
            print(f"[boruta] rodada {rodada + 1}/{n_rodadas}")

    confirma, rejeita, nivel_corrigido = limiares_binomiais(
        n_rodadas, len(variaveis), b["nivel_significancia"])
    tabela = pd.DataFrame({"variavel": variaveis, "vitorias": vitorias.values})
    tabela["n_rodadas"] = n_rodadas
    tabela["taxa_vitorias"] = tabela["vitorias"] / n_rodadas
    tabela["importancia_media"] = (importancia_acumulada / n_rodadas).values
    tabela["p_valor_confirmacao"] = binom.sf(tabela["vitorias"] - 1, n_rodadas, 0.5)
    tabela["p_valor_rejeicao"] = binom.cdf(tabela["vitorias"], n_rodadas, 0.5)
    tabela["decisao"] = np.select(
        [tabela["p_valor_confirmacao"] <= nivel_corrigido, tabela["p_valor_rejeicao"] <= nivel_corrigido],
        ["confirmada", "rejeitada"], default="tentativa")
    tabela.attrs.update(limiar_confirmacao=confirma, limiar_rejeicao=rejeita,
                        nivel_corrigido=nivel_corrigido)
    print(f"[boruta] {len(variaveis)} variáveis, nível corrigido = {nivel_corrigido:.2e}: "
          f"confirma com ≥ {confirma} vitórias, rejeita com ≤ {rejeita}")
    return tabela.sort_values(["vitorias", "importancia_media"], ascending=False).reset_index(drop=True)

## 8. Gráfico do Boruta e funil de seleção

**Gráfico.** Uma barra por variável com o número de vitórias em 50 rodadas, colorida pela decisão, e duas linhas verticais nos limiares de confirmação e de rejeição. As variáveis à direita da linha de confirmação seguem para o AutoML.

**Funil.** Quantas variáveis restam depois de cada filtro, para documentar a seleção:

```
candidatas 400 → faltantes 392 → quase constantes 371 → PSI 350 → poder 210 → redundância 150 → Boruta 40
```

In [ ]:
CORES_DECISAO = {"confirmada": "#2a78d6", "tentativa": "#eb6834", "rejeitada": "#b4b2a9"}


def grafico_boruta(tabela_boruta):
    """Barras horizontais de vitórias por variável, com os limiares do teste binomial."""
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch

    dados = tabela_boruta.sort_values("vitorias")
    altura = max(3.0, 0.22 * len(dados) + 1.5)
    figura, eixo = plt.subplots(figsize=(9, altura), dpi=130)
    eixo.barh(dados["variavel"], dados["vitorias"], height=0.65,
              color=[CORES_DECISAO[d] for d in dados["decisao"]])
    n_rodadas = int(dados["n_rodadas"].iloc[0])
    for limiar, rotulo in [(tabela_boruta.attrs.get("limiar_confirmacao"), "confirma"),
                           (tabela_boruta.attrs.get("limiar_rejeicao"), "rejeita")]:
        if limiar is not None:
            eixo.axvline(limiar, color="#52514e", lw=1, ls=(0, (4, 3)))
            eixo.text(limiar, 1.005, f"{rotulo} ({limiar})", transform=eixo.get_xaxis_transform(),
                      ha="center", va="bottom", fontsize=8, color="#52514e")
    eixo.set_xlim(0, n_rodadas)
    eixo.set_xlabel(f"vitórias contra a melhor sombra em {n_rodadas} rodadas")
    eixo.tick_params(axis="y", labelsize=8)
    eixo.spines[["top", "right"]].set_visible(False)
    eixo.grid(axis="x", color="#e4e3df", lw=0.8)
    eixo.set_axisbelow(True)
    eixo.legend(handles=[Patch(color=c, label=d) for d, c in CORES_DECISAO.items()],
                loc="upper left", bbox_to_anchor=(1.01, 1.0), frameon=False, fontsize=8)
    eixo.set_title("Boruta: vitórias de cada variável contra o ruído", loc="left", fontsize=11, pad=18)
    figura.tight_layout()
    return figura


def funil_selecao(tabela_prefiltros, tabela_boruta, tarefa="pd"):
    """Quantas variáveis restam após cada filtro (pandas), na ordem em que foram aplicados."""
    total = len(tabela_prefiltros)
    linhas = [("candidatas", total)]
    restantes = total
    poder = [("sem poder", "poder univariado"), ("teto da correlação", "teto da correlação")]
    if tarefa != "pd":
        poder = poder[::-1]                      # na LGD o teto vem antes do poder univariado
    for prefixo, rotulo in [("tipo", "tipo suportado"), ("faltantes", "faltantes"),
                            ("quase constante", "quase constantes"), ("instável", "estabilidade (PSI)"),
                            *poder, ("redundante", "redundância"), ("teto do Boruta", "teto do Boruta")]:
        restantes -= int(tabela_prefiltros["motivo"].str.startswith(prefixo).sum())
        linhas.append((f"após {rotulo}", restantes))
    linhas.append(("confirmadas no Boruta", int((tabela_boruta["decisao"] == "confirmada").sum())))
    return pd.DataFrame(linhas, columns=["etapa", "variaveis_restantes"])

## 9. Execução das etapas 2 e 3

`executa_etapa_selecao` encadeia tudo, exibe as tabelas e o gráfico, grava as tabelas Delta (`selecao_categorias`, `selecao_prefiltros`, `selecao_boruta`, `selecao_funil`) e registra o gráfico no MLflow (se houver execução ativa).

A base devolvida já tem as categorias raras agrupadas e as booleanas convertidas. Os cortes, as medianas e o mapa de categorias também são devolvidos, porque a etapa 4 (imputação) e a escoragem de bases novas precisam deles.

In [ ]:
def executa_etapa_selecao(spark, base, variaveis, cfg, gravar=True):
    """
    Etapas 2 (pré-filtros) e 3 (Boruta).

    Parâmetros
        base       saída "base" da etapa 1 (com a coluna particao).
        variaveis  saída "variaveis" da etapa 1.

    Devolve
        Dicionário com: base, variaveis_selecionadas, tipos, mapa_categorias,
        tabela_categorias, cortes, medianas, tabela_prefiltros, tabela_boruta,
        info_amostra_boruta, funil, figura_boruta.
    """
    tipos = classifica_tipos_variaveis(base, variaveis)
    base, tipos = padroniza_tipos(base, tipos)
    categoricas = [v for v, t in tipos.items() if t == "categorica"]
    numericas = [v for v, t in tipos.items() if t == "numerica"]
    print(f"[seleção] {len(numericas)} numéricas, {len(categoricas)} categóricas, "
          f"{sum(t == 'nao_suportada' for t in tipos.values())} de tipo não suportado")

    base, mapa_categorias, tabela_categorias = agrupa_categorias_raras(base, categoricas, cfg)
    # Sem nova gravação aqui: a entrada já foi guardada na etapa 1, e as conversões
    # de tipo e o agrupamento de categorias são contas simples aplicadas na leitura.
    # Regravar milhares de colunas custaria tanto quanto a gravação da etapa 1.
    base = persiste_se_possivel(base)
    cortes, medianas = calcula_cortes_e_medianas(base, numericas, cfg["prefiltros"]["n_faixas"],
                                                 tamanho_lote=cfg["tamanho_lote_variaveis"])

    tabela_prefiltros = aplica_prefiltros(base, tipos, cortes, medianas, cfg)
    aprovadas = tabela_prefiltros.loc[tabela_prefiltros["status"] == "mantida", "variavel"].tolist()
    exibe(tabela_prefiltros, "Pré-filtros")
    suspeitas = tabela_prefiltros.loc[tabela_prefiltros["suspeita_vazamento"], "variavel"].tolist()
    if suspeitas:
        print(f"[seleção] ATENÇÃO — suspeita de vazamento (conferir a origem): {suspeitas}")
    if not aprovadas:
        raise ValueError("Nenhuma variável passou pelos pré-filtros.")

    X, y, info_amostra = amostra_para_boruta(base, aprovadas, tipos, mapa_categorias, cfg)
    print(f"[boruta] amostra: {info_amostra['conta_tamanho']} | "
          f"{'treino inteiro' if info_amostra['usa_treino_inteiro'] else 'amostra'}: "
          f"{info_amostra['n_amostra_realizado']:,} contratos")
    tabela_boruta = executa_boruta(X, y, cfg)
    exibe(tabela_boruta, "Boruta")

    selecionadas = tabela_boruta.loc[tabela_boruta["decisao"] == "confirmada", "variavel"].tolist()
    if not selecionadas:
        raise ValueError("Nenhuma variável foi confirmada pelo Boruta.")
    funil = funil_selecao(tabela_prefiltros, tabela_boruta, cfg["tarefa"])
    exibe(funil, "Funil de seleção")

    figura = grafico_boruta(tabela_boruta)
    registra_figura_mlflow(figura, "selecao/boruta.png")

    if gravar:
        tabela_boruta_gravar = tabela_boruta.assign(**{k: str(v) for k, v in tabela_boruta.attrs.items()})
        tabela_boruta_gravar.attrs = {}
        for nome, tabela in [("selecao_categorias", tabela_categorias),
                             ("selecao_prefiltros", tabela_prefiltros),
                             ("selecao_boruta", tabela_boruta_gravar),
                             ("selecao_funil", funil)]:
            if len(tabela):
                grava_tabela_delta(spark, tabela, nome, cfg)

    return {
        "base": base, "variaveis_selecionadas": selecionadas, "tipos": tipos,
        "mapa_categorias": mapa_categorias, "tabela_categorias": tabela_categorias,
        "cortes": cortes, "medianas": medianas,
        "tabela_prefiltros": tabela_prefiltros, "tabela_boruta": tabela_boruta,
        "info_amostra_boruta": info_amostra, "funil": funil, "figura_boruta": figura,
    }

## 10. Exemplo de uso

```python
etapa1 = executa_etapa_entrada(spark, base_entrada, cfg)
etapa2 = executa_etapa_selecao(spark, etapa1["base"], etapa1["variaveis"], cfg)

etapa2["variaveis_selecionadas"]   # seguem para o AutoML (notebook 03)
etapa2["figura_boruta"]            # gráfico do Boruta
```